# Task 5 — Marketplace Integration & Company Portal v1 (Matching Validation)
**Focus:** Validate matching on the integrated flow — end-to-end: company posts → student applies → company shortlists.

This is a **standalone notebook**. It re-loads students.csv / jobs.csv / matches.csv fresh and
re-defines the ranking + explanation functions needed (built in Tasks 3 & 4), so it runs
independently. No need to run any other notebook first.

## Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import json
from sklearn.metrics import precision_score, recall_score, confusion_matrix
print('Libraries loaded')

Libraries loaded


## Cell 2 — Load data

In [2]:
students = pd.read_csv('../datasets/students.csv')
jobs     = pd.read_csv('../datasets/jobs.csv')
matches  = pd.read_csv('../datasets/matches.csv')
print('Students:', students.shape, '| Jobs:', jobs.shape, '| Matches:', matches.shape)

Students: (20, 7) | Jobs: (9, 7) | Matches: (180, 6)


## Cell 3 — Core helper functions (skills, vectors, thresholds)

In [3]:
def parse_skills(skill_str):
    result = {}
    if pd.isna(skill_str) or str(skill_str).strip() == '':
        return result
    for part in str(skill_str).split(','):
        part = part.strip()
        if ':' in part:
            skill, score = part.split(':', 1)
            try:
                result[skill.strip()] = int(score.strip())
            except ValueError:
                pass
    return result


def compute_skill_overlap(student_skills_str, job_skills_str):
    student_skills = parse_skills(student_skills_str)
    required_skills = parse_skills(job_skills_str)
    if not required_skills:
        return 0, 0.0
    met = sum(1 for skill, req in required_skills.items() if student_skills.get(skill, 0) >= req)
    return met, round(met / len(required_skills), 3)


def compute_experience_gap(internship_months, min_exp_years):
    return round(internship_months / 6.0 - min_exp_years, 2)


def validate_thresholds(student_id, job_id):
    student = students[students['student_id'] == student_id].iloc[0]
    job     = jobs[jobs['job_id'] == job_id].iloc[0]
    student_skills  = parse_skills(student['skills'])
    required_skills = parse_skills(job['required_skills'])
    results = {}
    for skill, required_level in required_skills.items():
        student_level = student_skills.get(skill, 0)
        results[skill] = {
            'required_level': required_level,
            'student_level': student_level,
            'passed': bool(student_level >= required_level)
        }
    overall_passed = all(r['passed'] for r in results.values()) if results else False
    return results, overall_passed


all_skills = set()
for s in students['skills'].dropna():
    all_skills.update(parse_skills(s).keys())
for s in jobs['required_skills'].dropna():
    all_skills.update(parse_skills(s).keys())
MASTER_SKILLS = sorted(all_skills)


def skills_to_vector(skill_str):
    skill_dict = parse_skills(skill_str)
    return np.array([skill_dict.get(skill, 0) for skill in MASTER_SKILLS])


students['skill_vector'] = students['skills'].apply(skills_to_vector)
jobs['threshold_vector']  = jobs['required_skills'].apply(skills_to_vector)


def cosine_similarity_manual(vec1, vec2):
    dot = np.dot(vec1, vec2)
    n1, n2 = np.linalg.norm(vec1), np.linalg.norm(vec2)
    return 0.0 if n1 == 0 or n2 == 0 else dot / (n1 * n2)


print(f'Master skill list ({len(MASTER_SKILLS)} skills):', MASTER_SKILLS)
print('Helper functions ready')

Master skill list (35 skills): ['.NET', 'API', 'AWS', 'Android', 'C#', 'C++', 'CSS', 'Cybersecurity', 'DSA', 'Data Science', 'Django', 'Docker', 'Excel', 'Flask', 'Git', 'HTML', 'Java', 'JavaScript', 'Kotlin', 'Linux', 'ML', 'Microservices', 'MongoDB', 'Networking', 'Node', 'Pandas', 'PowerBI', 'Python', 'REST', 'React', 'SQL', 'Spring', 'Statistics', 'Tableau', 'TensorFlow']
Helper functions ready


## Cell 4 — Ranking function: Candidates for a Job (reused from Task 3)

In [4]:
def rank_students_for_job(job_id, top_n=None):
    """Rank all students for a job using cosine similarity on skill vectors."""
    j_vec = jobs[jobs['job_id'] == job_id]['threshold_vector'].iloc[0]
    results = []
    for _, student in students.iterrows():
        sim = cosine_similarity_manual(student['skill_vector'], j_vec)
        _, overall_passed = validate_thresholds(student['student_id'], job_id)
        results.append({
            'student_id': student['student_id'],
            'preferred_role': student['preferred_role'],
            'match_score': round(sim, 3),
            'threshold_passed': overall_passed
        })
    ranked = pd.DataFrame(results).sort_values('match_score', ascending=False).reset_index(drop=True)
    ranked.index += 1
    return ranked.head(top_n) if top_n else ranked


print('rank_students_for_job() ready')
print(rank_students_for_job(jobs['job_id'].iloc[0], top_n=5))

rank_students_for_job() ready
   student_id  preferred_role  match_score  threshold_passed
1           6    Data Analyst        0.985             False
2           1    Data Analyst        0.856              True
3          19    Data Analyst        0.737             False
4          11  Data Scientist        0.723             False
5          13    Data Analyst        0.636             False


## Cell 5 — Explanation payload function (reused from Task 4)

In [5]:
def build_explanation_payload(student_id, job_id):
    edge_case_flags = []
    student_match = students[students['student_id'] == student_id]
    job_match = jobs[jobs['job_id'] == job_id]

    if student_match.empty or job_match.empty:
        return {
            'student_id': int(student_id), 'job_id': int(job_id), 'match_score': 0.0,
            'decision': 'ERROR', 'reason': 'Student or job not found.',
            'edge_case_flags': ['student_or_job_not_found']
        }

    student = student_match.iloc[0]
    job = job_match.iloc[0]
    required_skills = parse_skills(job['required_skills'])
    threshold_results, _ = validate_thresholds(student_id, job_id)

    skill_breakdown = [
        {'skill': s, 'required': int(r['required_level']), 'student_has': int(r['student_level']), 'passed': bool(r['passed'])}
        for s, r in threshold_results.items()
    ]
    missing_skills = [s['skill'] for s in skill_breakdown if not s['passed']]

    overlap_count, overlap_ratio = compute_skill_overlap(student['skills'], job['required_skills'])
    exp_gap = compute_experience_gap(student['internship_months'], job['min_experience_years'])
    exp_passed = bool(exp_gap >= -0.5)

    decision = 'MATCH' if (overlap_ratio >= 0.6 and exp_passed) else 'NO_MATCH'

    if decision == 'MATCH':
        reason = f"Meets {overlap_count}/{len(required_skills)} required skills and experience is sufficient."
    else:
        reasons = []
        if overlap_ratio < 0.6:
            reasons.append(f"only meets {overlap_count}/{len(required_skills)} required skills")
        if not exp_passed:
            reasons.append(f"experience is {abs(exp_gap)} years short")
        reason = 'Not a match: ' + '; '.join(reasons) + '.'

    return {
        'student_id': int(student_id), 'job_id': int(job_id),
        'match_score': float(round(overlap_ratio, 3)), 'decision': decision,
        'skill_breakdown': skill_breakdown, 'missing_skills': missing_skills,
        'reason': reason, 'edge_case_flags': edge_case_flags
    }


print('build_explanation_payload() ready')

build_explanation_payload() ready


## Cell 6 — Simulate the Integrated Flow

**Step 1:** Company posts a job (pick a real job)
**Step 2:** Students apply (all students are treated as applicants — open marketplace)
**Step 3:** Company shortlists (take top-K ranked candidates)
**Step 4:** Each shortlisted candidate gets an explanation payload

In [6]:
def simulate_company_flow(job_id, shortlist_size=3):
    """
    Simulates: company posts job -> students apply -> company shortlists top-K -> explain each.
    """
    job = jobs[jobs['job_id'] == job_id].iloc[0]

    # Step 1: company posts job (already exists in jobs.csv)
    # Step 2: all students 'apply' (open marketplace assumption)
    applicants = students['student_id'].tolist()

    # Step 3: rank applicants, shortlist top-K
    ranked = rank_students_for_job(job_id, top_n=shortlist_size)
    shortlist = ranked['student_id'].tolist()

    # Step 4: explain each shortlisted candidate
    explanations = [build_explanation_payload(sid, job_id) for sid in shortlist]

    return {
        'job_id': int(job_id),
        'job_title': job['job_title'],
        'company_name': job['company_name'],
        'total_applicants': len(applicants),
        'shortlist_size': shortlist_size,
        'shortlisted_students': shortlist,
        'explanations': explanations
    }


# Run on one job as a live demo
demo_job_id = jobs['job_id'].iloc[0]
flow_result = simulate_company_flow(demo_job_id, shortlist_size=3)
print(json.dumps(flow_result, indent=2))

{
  "job_id": 101,
  "job_title": "Data Analyst",
  "company_name": "TechNova",
  "total_applicants": 20,
  "shortlist_size": 3,
  "shortlisted_students": [
    6,
    1,
    19
  ],
  "explanations": [
    {
      "student_id": 6,
      "job_id": 101,
      "match_score": 0.667,
      "decision": "MATCH",
      "skill_breakdown": [
        {
          "skill": "Python",
          "required": 70,
          "student_has": 65,
          "passed": false
        },
        {
          "skill": "SQL",
          "required": 60,
          "student_has": 60,
          "passed": true
        },
        {
          "skill": "Excel",
          "required": 50,
          "student_has": 70,
          "passed": true
        }
      ],
      "missing_skills": [
        "Python"
      ],
      "reason": "Meets 2/3 required skills and experience is sufficient.",
      "edge_case_flags": []
    },
    {
      "student_id": 1,
      "job_id": 101,
      "match_score": 1.0,
      "decision": "MATCH",
     

## Cell 7 — Validate the Flow Across MULTIPLE Jobs (real-data scale)

Run the full company flow for every job, not just one. This proves the ranking
and shortlisting hold up at scale, not on a single happy-path example.

In [7]:
all_flow_results = []
for job_id in jobs['job_id'].unique():
    result = simulate_company_flow(job_id, shortlist_size=3)
    all_flow_results.append(result)

print(f'Simulated end-to-end flow for {len(all_flow_results)} jobs')
for r in all_flow_results:
    print(f"Job {r['job_id']} ({r['job_title']}): shortlisted {r['shortlisted_students']}")

Simulated end-to-end flow for 9 jobs
Job 101 (Data Analyst): shortlisted [6, 1, 19]
Job 102 (Backend Developer): shortlisted [14, 2, 7]
Job 103 (ML Engineer): shortlisted [3, 11, 15]
Job 104 (BI Analyst): shortlisted [4, 13, 6]
Job 105 (Frontend Developer): shortlisted [5, 18, 12]
Job 106 (Cloud Engineer): shortlisted [10, 17, 1]
Job 107 (Security Analyst): shortlisted [17, 10, 1]
Job 108 (Android Developer): shortlisted [16, 14, 7]
Job 109 (Software Engineer): shortlisted [20, 13, 19]


## Cell 8 — Validate Shortlist Quality Against True Labels

For every job, check: of the students we shortlisted, how many are TRUE good matches
(label=1 in matches.csv)? This is Precision@K and Recall@K, computed across ALL jobs —
the actual 'rankings validated end-to-end' deliverable.

In [8]:
def validate_shortlist_quality(k=3):
    precisions, recalls = [], []
    per_job_results = []

    for job_id in jobs['job_id'].unique():
        true_positive_students = set(
            matches[(matches['job_id'] == job_id) & (matches['label'] == 1)]['student_id']
        )
        if not true_positive_students:
            continue

        ranked = rank_students_for_job(job_id, top_n=k)
        shortlisted = set(ranked['student_id'])

        hits = len(shortlisted & true_positive_students)
        precision = hits / k
        recall = hits / len(true_positive_students)

        precisions.append(precision)
        recalls.append(recall)
        per_job_results.append({
            'job_id': job_id, 'true_positives_available': len(true_positive_students),
            'shortlist_hits': hits, 'precision_at_k': round(precision, 3), 'recall_at_k': round(recall, 3)
        })

    return pd.DataFrame(per_job_results), np.mean(precisions), np.mean(recalls)


per_job_df, avg_precision, avg_recall = validate_shortlist_quality(k=3)
print('=== PER-JOB SHORTLIST VALIDATION (K=3) ===')
print(per_job_df.to_string(index=False))
print()
print(f'AVERAGE Precision@3 across all jobs: {avg_precision:.3f}')
print(f'AVERAGE Recall@3 across all jobs   : {avg_recall:.3f}')

=== PER-JOB SHORTLIST VALIDATION (K=3) ===
 job_id  true_positives_available  shortlist_hits  precision_at_k  recall_at_k
    101                         8               3           1.000        0.375
    102                         3               3           1.000        1.000
    103                         1               1           0.333        1.000
    104                         4               3           1.000        0.750
    105                         2               2           0.667        1.000
    106                         1               1           0.333        1.000
    107                         1               1           0.333        1.000
    108                         1               1           0.333        1.000
    109                         1               1           0.333        1.000

AVERAGE Precision@3 across all jobs: 0.593
AVERAGE Recall@3 across all jobs   : 0.903


## Cell 9 — Consistency Check (not independent validation)

**Important honesty note:** `matches.csv` labels were generated using the same rule
(`overlap_ratio >= 0.6 AND exp_gap >= -0.5`) that `build_explanation_payload()` uses to
decide MATCH/NO_MATCH. So this check only confirms the two functions agree with each
other — it is **not** an independent accuracy measurement. The real validation of
ranking quality is Cell 8 (Precision@K / Recall@K), since ranking by cosine similarity
is a genuinely different method than the labeling rule.

We still run this as a sanity check — if it failed, that would mean a bug in one of
the two functions.

In [9]:
# SANITY CHECK ONLY — see note in markdown above about circularity
y_true = matches['label'].values
y_pred = []
for _, row in matches.iterrows():
    payload = build_explanation_payload(row['student_id'], row['job_id'])
    y_pred.append(1 if payload['decision'] == 'MATCH' else 0)

cm = confusion_matrix(y_true, y_pred)
print('Confusion Matrix (sanity check — same rule on both sides):')
print(cm)

if cm.size == 4:
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
else:
    fpr = 0.0
print(f'\nNote: this will show 1.0 precision/recall because the decision rule and the')
print(f'label rule are the same formula. This confirms no bug exists, but is NOT proof')
print(f'of real-world accuracy. Refer to Cell 8 Precision@K for genuine ranking validation.')

Confusion Matrix (sanity check — same rule on both sides):
[[158   0]
 [  0  22]]

Note: this will show 1.0 precision/recall because the decision rule and the
label rule are the same formula. This confirms no bug exists, but is NOT proof
of real-world accuracy. Refer to Cell 8 Precision@K for genuine ranking validation.


## Cell 10 — Edge Case: Job With No Qualifying Candidates

What happens when a job's thresholds are so strict that nobody passes? The flow
should still return a shortlist (even if low-confidence) and not crash.

In [10]:
# Find a job with the fewest qualifying candidates to demonstrate this edge case
qualifying_counts = []
for job_id in jobs['job_id'].unique():
    _, overall = None, None
    count = sum(1 for sid in students['student_id'] if validate_thresholds(sid, job_id)[1])
    qualifying_counts.append((job_id, count))

qualifying_counts.sort(key=lambda x: x[1])
hardest_job_id, hardest_count = qualifying_counts[0]

print(f'Job {hardest_job_id} has only {hardest_count} fully-qualifying students out of {len(students)}')
print()
edge_flow = simulate_company_flow(hardest_job_id, shortlist_size=3)
print(f"Shortlist still returned: {edge_flow['shortlisted_students']}")
print('(System degrades gracefully — returns best-available candidates, does not crash or return nothing.)')

Job 101 has only 1 fully-qualifying students out of 20

Shortlist still returned: [6, 1, 19]
(System degrades gracefully — returns best-available candidates, does not crash or return nothing.)


## Cell 11 — Final Validation Report (Hand-off: Matching Go-Ahead)

In [11]:
validation_report = {
    'jobs_tested': len(jobs),
    'students_in_pool': len(students),
    'total_pairs_evaluated': len(matches),
    'avg_precision_at_3': round(avg_precision, 3),
    'avg_recall_at_3': round(avg_recall, 3),
    'sanity_check_consistency': 'PASS (decision logic matches label rule with no bugs)',
    'edge_case_handled': 'low-qualifying-candidate job tested, no crash, graceful shortlist returned',
    'verdict': 'GO' if avg_precision >= 0.3 and avg_recall >= 0.3 else 'NEEDS_REVIEW'
}

print('=== FINAL MATCHING VALIDATION REPORT ===')
print(json.dumps(validation_report, indent=2))
print()
print('Verdict based on Precision@3/Recall@3 (genuine ranking validation), not the')
print('circular sanity check. This is the "Matching go-ahead" hand-off.')

=== FINAL MATCHING VALIDATION REPORT ===
{
  "jobs_tested": 9,
  "students_in_pool": 20,
  "total_pairs_evaluated": 180,
  "avg_precision_at_3": 0.593,
  "avg_recall_at_3": 0.903,
  "sanity_check_consistency": "PASS (decision logic matches label rule with no bugs)",
  "edge_case_handled": "low-qualifying-candidate job tested, no crash, graceful shortlist returned",
  "verdict": "GO"
}

Verdict based on Precision@3/Recall@3 (genuine ranking validation), not the
circular sanity check. This is the "Matching go-ahead" hand-off.


## Cell 12 — Save Full Flow Results (hand-off artifact)

In [12]:
output = {
    'validation_report': validation_report,
    'per_job_results': per_job_df.to_dict(orient='records'),
    'sample_flow_demo': flow_result
}

with open('task5_validation_report.json', 'w') as f:
    json.dump(output, f, indent=2)

print('Saved task5_validation_report.json')
print('This is the hand-off artifact confirming the matching system is validated end-to-end.')

Saved task5_validation_report.json
This is the hand-off artifact confirming the matching system is validated end-to-end.
